# 13. OOP Basics (5+ Years Interview Guide)
Class blueprints, object instantiation mechanics, constructor lifecycle (__new__ vs __init__), name mangling, class vs instance variables, and attribute reflection.

### Key 5-Year Interview Concepts Covered:
- **Constructor Protocol**: `__new__` (allocates the instance) vs `__init__` (initializes instance state).
- **Access Control & Name Mangling**: Public, protected (`_var`), and private (`__var` -> `_ClassName__var`).
- **Class Variables vs Instance Variables**: Shared class memory vs independent `self.__dict__` instance namespaces.
- **Attribute Reflection**: Dynamic inspection and manipulation via `getattr()`, `setattr()`, `hasattr()`, and `delattr()`.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [1]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

Using CSV file path: data/raw_transactions.csv


### 1. Class Definitions & Types
**Explanation**: A `class` in Python is an object itself (an instance of the metaclass `type`). Defining a class executes its body statements in a temporary namespace dictionary, which is then passed to `type(name, bases, dict)` to construct the class object in memory.

**Syntax**: `class ClassName: ...`

In [2]:
class EnterpriseRecord:
    pass
print(EnterpriseRecord)

<class '__main__.EnterpriseRecord'>


### 2. Object Instantiation Mechanics
**Explanation**: Calling `obj = ClassName(*args)` triggers a two-step lifecycle: 1. `ClassName.__new__(cls, *args)` is called to allocate the raw object in heap memory and return the new instance; 2. `instance.__init__(*args)` is called to initialize instance attributes on `self`.

**Syntax**: `instance = ClassName(arg1, arg2)`

In [3]:
class EnterpriseRecord:
    pass
record_instance = EnterpriseRecord()
print(record_instance)

### 3. Constructor Initializations (`__init__`)
**Explanation**: The `__init__` method initializes the newly created instance. It must always return `None` (returning any non-None value raises a `TypeError`). All instance attributes attached to `self` are stored in the instance's private `__dict__`.

**Syntax**: `def __init__(self, param1): self.param1 = param1`

In [4]:
class EnterpriseRecord:
    def __init__(self, initialization_value): self.initialization_value = initialization_value
print(EnterpriseRecord(5).initialization_value)

5


### 4. Instance Attributes & The `self` Parameter
**Explanation**: In Python, `self` represents the active instance. When calling `obj.method()`, Python automatically converts it to `ClassName.method(obj)`. Unlike other languages where `this` is a keyword, `self` is explicitly declared as the first parameter of instance methods by convention.

**Syntax**: `self.attribute_name = value`

In [5]:
class EnterpriseRecord:
    def get_self_reference(self): return self
print(EnterpriseRecord().get_self_reference())

### 5. Private Attributes & Name Mangling (`__var`)
**Explanation**: Prefixing an attribute with two leading underscores (and at most one trailing underscore, e.g. `__secret`) triggers CPython's name mangling mechanism. The identifier is internally transformed to `_ClassName__secret` in `__dict__`. This prevents subclasses from accidentally overriding internal parent attributes.

**Syntax**: `self.__private_var = value  # Mangled to _ClassName__private_var`

In [6]:
class EnterpriseRecord:
    def __init__(self): self.__private_value = 10
try:
    print(EnterpriseRecord().__private_value)
except AttributeError as error_message:
    print('Mangled attribute error:', error_message)

Mangled attribute error: 'EnterpriseRecord' object has no attribute '__private_value'


### 6. Protected Attributes Convention (`_var`)
**Explanation**: Prefixing an attribute with a single leading underscore `_status` is an industry convention signaling that the attribute is protected/internal. Python does not enforce access restrictions at the interpreter level; it relies on developer convention ('we are all consenting adults here').

**Syntax**: `self._protected_var = value  # Internal API convention`

In [7]:
class EnterpriseRecord:
    def __init__(self): self._protected_value = 10
print('Protected value accessed:', EnterpriseRecord()._protected_value)

Protected value accessed: 10


### 7. Class Variables vs Instance Variables
**Explanation**: Class variables are defined directly inside the class body and are shared across all instances in the class dictionary `ClassName.__dict__`. Instance variables are attached to `self` and reside in each instance's individual `self.__dict__`.

**Syntax**: `class Account: interest_rate = 0.05  # Shared class variable`

In [8]:
class EnterpriseRecord:
    shared_class_variable = 0
    def __init__(self): self.unique_instance_variable = 1
instance_one = EnterpriseRecord()
instance_two = EnterpriseRecord()
print('Class Var Match:', instance_one.shared_class_variable == instance_two.shared_class_variable)

Class Var Match: True


### 8. Modifying Shared Class Variables Pitfall
**Explanation**: If you assign to a class variable via an instance `obj.interest_rate = 0.08`, Python does NOT modify the shared class variable; instead, it creates a new instance variable on `obj` that shadows the class variable! To modify the shared value for all instances, modify it via the class: `Account.interest_rate = 0.08`.

**Syntax**: `ClassName.class_var = new_val  # Modifies for all` / `obj.class_var = val  # Shadows on instance`

In [9]:
class EnterpriseRecord:
    shared_class_variable = 1
EnterpriseRecord.shared_class_variable = 2
print(EnterpriseRecord().shared_class_variable)

2


### 9. Instance Methods & Bound Method Objects
**Explanation**: Accessing a function through an instance `obj.method` returns a 'bound method' object that bundles the function pointer with `self`. When called, the bound method automatically passes `obj` as the first argument.

**Syntax**: `def method_name(self, arg1): return self.attr + arg1`

In [10]:
class EnterpriseRecord:
    def execute_record_call(self): return 'Ok'
print(EnterpriseRecord().execute_record_call())

Ok


### 10. Class Interface Queries (`isinstance` & `issubclass`)
**Explanation**: `isinstance(obj, ClassName)` checks if an object is an instance of a class or any of its subclasses. `issubclass(ChildClass, ParentClass)` tests class inheritance relationships directly.

**Syntax**: `isinstance(obj, (ClassA, ClassB))` / `issubclass(Sub, Base)`

In [11]:
class BaseRecord: pass
class SubRecord(BaseRecord): pass
print('Issubclass?:', issubclass(SubRecord, BaseRecord))

Issubclass?: True


### 11. Instance Namespace Dictionaries (`__dict__`)
**Explanation**: Every standard Python instance stores its dynamic attributes in an internal dictionary `obj.__dict__`. Accessing `obj.name` looks up `'name'` in `obj.__dict__`. Directly inspecting or modifying `__dict__` allows low-level serialization and dynamic attribute injection.

**Syntax**: `obj.__dict__['dynamic_attr'] = 100` / `print(obj.__dict__)`

In [12]:
class EnterpriseRecord:
    def __init__(self): self.attribute_x = 1
print(EnterpriseRecord().__dict__)

{'attribute_x': 1}


### 12. Object Directory Inspection (`dir()`)
**Explanation**: `dir(obj)` returns a sorted list of all valid attribute and method names available on the object, including inherited methods, class variables, and magic dunder methods by querying `__dir__()`.

**Syntax**: `all_attributes = dir(object_instance)`

In [13]:
class EnterpriseRecord: pass
print('__init__' in dir(EnterpriseRecord()))

True


### 13. Dynamic Attribute Assignment
**Explanation**: Because instance namespaces are dictionaries, Python allows attaching new attributes to existing instances at runtime: `obj.custom_flag = True`. While flexible, dynamically adding attributes outside `__init__` should be used sparingly in production to maintain predictable object schemas.

**Syntax**: `obj.new_field = dynamic_value`

In [14]:
class EnterpriseRecord: pass
record_instance = EnterpriseRecord()
record_instance.temporary_field = 99
print(record_instance.temporary_field)

99


### 14. Deleting Object Attributes (`delattr` & `del`)
**Explanation**: `del obj.attribute_name` or `delattr(obj, 'attribute_name')` removes the attribute from `obj.__dict__`. Subsequent access will trigger an `AttributeError` unless a class variable with the same name exists to fall back on.

**Syntax**: `del obj.attribute_name` / `delattr(obj, 'attribute_name')`

In [15]:
class EnterpriseRecord:
    def __init__(self): self.attribute_x = 1
record_instance = EnterpriseRecord()
del record_instance.attribute_x
try: print(record_instance.attribute_x)
except AttributeError as error_message: print('Deleted safely:', error_message)

Deleted safely: 'EnterpriseRecord' object has no attribute 'attribute_x'


### 15. Attribute Reflection (`getattr`, `setattr`, `hasattr`)
**Explanation**: Attribute reflection functions enable programmatic attribute manipulation by string name: `getattr(obj, 'name', default)` reads an attribute safely; `setattr(obj, 'name', val)` sets it; `hasattr(obj, 'name')` checks if it exists. This is foundational for building ORMs, serializers, and plugin systems.

**Syntax**: `val = getattr(obj, 'field', None)` / `setattr(obj, 'field', val)` / `hasattr(obj, 'field')`

In [16]:
class EnterpriseRecord: pass
record_instance = EnterpriseRecord()
setattr(record_instance, 'reflected_value', 50)
print('Has field?:', hasattr(record_instance, 'reflected_value'), 'Value:', getattr(record_instance, 'reflected_value'))

Has field?: True Value: 50


## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Encapsulating financial transaction models, private fee calculations, and dynamic attribute mapping for ORM models.


In [17]:
# Solution:
class TransactionRecord:
    def __init__(self, tx_id, amount):
        self.tx_id = tx_id
        self.__amount = amount
        
    def get_amount(self): return self.__amount
    def set_amount(self, val): self.__amount = val

with open(csv_path, 'r') as f:
    f.readline()
    row = f.readline().strip().split(',')
    tr = TransactionRecord(row[0], float(row[3]))
    print('Reflected has amount?:', hasattr(tr, '_TransactionRecord__amount'))
    print('Amount via getter:', tr.get_amount())


Reflected has amount?: True
Amount via getter: 1216.33


### Q2: Object Reflection & Namespace Introspection
**Explanation**: **Scenario**: Build a `TransactionRecord` class encapsulating raw transaction fields, parse dataset rows, and use `getattr`/`setattr` reflection to dynamically transform fields.

**Syntax**: `for k, v in row.items(): setattr(record, k, v)`

In [18]:
# Solution:
tr = TransactionRecord('TX100', 120.0)
print('Keys in __dict__ namespace:', list(tr.__dict__.keys()))


Keys in __dict__ namespace: ['tx_id', '_TransactionRecord__amount']
